# Radiomics Feature Extraction Test
This is a simple script to test extracting texture features from a single ultrasound image using PyRadiomics. 
We are grabbing one image from our data, formatting it correctly, and letting the library do the math.

In [1]:
import os
import cv2
import pydicom
import numpy as np
import SimpleITK as sitk
from radiomics import featureextractor

print("Everything imported successfully!")

Everything imported successfully!


In [2]:
# 1. Define folder paths
raw_data_folder = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")
mask_folder = os.path.join("..", "data", "PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED_ERODED_K3_I1", "masks")

# 2. We will test on just one patient
test_patient_id = "01_01"

print("Testing on patient:", test_patient_id)

Testing on patient: 01_01


In [3]:
# 3. Load the original DICOM image first
patient_folder = os.path.join(raw_data_folder, test_patient_id)

# Find the date folder inside the patient folder
list_of_folders = []
for f in os.listdir(patient_folder):
    if not f.startswith("."):
        list_of_folders.append(f)
        
date_folder_name = list_of_folders[0]
date_folder_path = os.path.join(patient_folder, date_folder_name)

# Find the DICOM file inside the date folder
list_of_files = []
for f in os.listdir(date_folder_path):
    if not f.startswith("."):
        list_of_files.append(f)
        
dicom_file_name = list_of_files[0]
dicom_full_path = os.path.join(date_folder_path, dicom_file_name)

print("Reading image from:", dicom_full_path)

# Read DICOM and get pixels
dicom_dataset = pydicom.dcmread(dicom_full_path)
image_pixels = dicom_dataset.pixel_array

# Convert to grayscale if it is in color (has 3 channels)
if len(image_pixels.shape) == 3:
    gray_image = cv2.cvtColor(image_pixels, cv2.COLOR_RGB2GRAY)
else:
    gray_image = image_pixels

print("Original image shape is:", gray_image.shape)

Reading image from: ../data/PANCREAS_2/PANCREAS_2/01_01/20181008/1.3.6.1.4.1.14519.5.2.1.9688.9989.203122483102738164055454061479
Original image shape is: (768, 1024)


In [4]:
# 4. Load our prepared binary mask
mask_file_name = test_patient_id + "_mask_eroded_k3_i1.png"
mask_full_path = os.path.join(mask_folder, mask_file_name)

print("Reading mask from:", mask_full_path)

mask_pixels = cv2.imread(mask_full_path, cv2.IMREAD_GRAYSCALE)

# Fix the mask so the region is exactly 1 (PyRadiomics rule)
# Right now the white part is 255.
mask_binary = np.zeros_like(mask_pixels, dtype=np.uint8)
mask_binary[mask_pixels > 0] = 1

print("Mask shape is:", mask_binary.shape)

Reading mask from: ../data/PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED_ERODED_K3_I1/masks/01_01_mask_eroded_k3_i1.png
Mask shape is: (768, 1024)


In [5]:
# 5. Check if the mask and image are different shapes 
# (Reusing our safeguard logic from before)
if gray_image.shape != mask_binary.shape:
    print("Handling shape mismatch!")
    mask_binary = cv2.resize(mask_binary, (gray_image.shape[1], gray_image.shape[0]), interpolation=cv2.INTER_NEAREST)

# 6. Convert numpy arrays into SimpleITK objects 
# (This is just a format that PyRadiomics understands)
sitk_image = sitk.GetImageFromArray(gray_image)
sitk_mask = sitk.GetImageFromArray(mask_binary)

print("Variables converted correctly to SimpleITK.")

Variables converted correctly to SimpleITK.


In [6]:
# 7. Set up the extractor object
extractor = featureextractor.RadiomicsFeatureExtractor()

# Disable all the complex default features we don't need
extractor.disableAllFeatures()

# Enable only texture-based features (GLCM) and basic First Order stats
extractor.enableFeatureClassByName('glcm')
extractor.enableFeatureClassByName('firstorder')

print("Starting feature extraction... This might take a few seconds.")
features = extractor.execute(sitk_image, sitk_mask)

print("Done! Extraction complete.\n")

# 8. Print out what we found step by step
print("---- EXTRACTED FEATURES ----")
for feature_name in features.keys():
    # Skip the system info metadata that prints at the top
    if "diagnostics_" in feature_name:
        continue
    
    feature_value = features[feature_name]
    print(feature_name, "=", feature_value)

Starting feature extraction... This might take a few seconds.


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


Done! Extraction complete.

---- EXTRACTED FEATURES ----
original_glcm_Autocorrelation = 5.661871105266115
original_glcm_ClusterProminence = 8.240012199059123
original_glcm_ClusterShade = 0.6576582282597085
original_glcm_ClusterTendency = 1.6023945007303289
original_glcm_Contrast = 0.15329843982176602
original_glcm_Correlation = 0.8252769455165221
original_glcm_DifferenceAverage = 0.15318674809748686
original_glcm_DifferenceEntropy = 0.6084998610630182
original_glcm_DifferenceVariance = 0.12823788737201744
original_glcm_Id = 0.9234252412386365
original_glcm_Idm = 0.9234177951236845
original_glcm_Idmn = 0.9941047949542632
original_glcm_Idn = 0.974471534643854
original_glcm_Imc1 = -0.5202324005822675
original_glcm_Imc2 = 0.8772067395594323
original_glcm_InverseVariance = 0.1530890178387426
original_glcm_JointAverage = 2.302085111131351
original_glcm_JointEnergy = 0.33278045330460526
original_glcm_JointEntropy = 2.137124619153264
original_glcm_MCC = 0.831409611132486
original_glcm_Maximum